In [72]:
# Testing the geoapify replacement for google maps api since osm is washed
import pandas as pd
import numpy as np
import requests
import time
import json
from dotenv import load_dotenv, dotenv_values
from pathlib import Path
import os
from fuzzywuzzy import fuzz

In [73]:
env_path = Path(os.path.abspath('')).resolve().parent / "secrets.env"
env_vars = dotenv_values(env_path)
geoapify_key = env_vars['GEOAPIFY_KEY']

In [ ]:
df = pd.read_csv("data_2026-03-02_2026-03-08.csv", parse_dates = ['approved_date'])
# Filter out bad coordinates
df = df.dropna(subset = ['longitude', 'latitude'])
df = df[(df['longitude'] != 0) & (df['latitude'] != 0)]

# Clean and combine address data for later processing
df['street_name'] = df['street_name'].apply(lambda x : x.title())
df['address'] = df.apply(lambda x : str(x['house_number']) + ' ' + x['street_name'] ,axis = 1)

# Turn dates from timestamp level to date level
df['approved_date'] = df['approved_date'].apply(lambda x : pd.Timestamp(year = x.year, month = x.month, day = x.day))

# df = df.iloc[201:300]

df = df[
    ['inspection_type',
     'job_id',
     'job_progress',
     'house_number',
     'street_name',
     'address',
     'zip_code',
     'latitude',
     'longitude',
     'result',
     'approved_date', # Please change this to inspection date
     'nta'
     ]
]

df.columns = df.columns.str.lower() + '_interdata'
df.head()

,inspection_type_interdata,job_id_interdata,job_progress_interdata,house_number_interdata,street_name_interdata,address_interdata,zip_code_interdata,latitude_interdata,longitude_interdata,result_interdata,approved_date_interdata,nta_interdata
0,Compliance,PC8654522,2,246,East 5 Street,246 East 5 Street,10003.0,40.727247,-73.990136,Rat Activity,2026-03-03,East Village
2,Initial,PC8673353,1,88-47,187 Street,88-47 187 Street,11423.0,40.713865,-73.774740,Rat Activity,2026-03-03,Hollis
3,Initial,PC8671158,1,675,Riverside Drive,675 Riverside Drive,10031.0,40.826757,-73.952242,Failed for Rat Act,2026-03-04,Hamilton Heights-Sugar Hill
4,Initial,PC8672570,1,1603,President Street,1603 President Street,11213.0,40.667220,-73.935026,Failed for Other R,2026-03-03,Crown Heights (South)
5,Initial,PC8662766,1,768,Putnam Avenue,768 Putnam Avenue,11221.0,40.686198,-73.931088,Rat Activity,2026-03-03,Bedford-Stuyvesant (East)


In [75]:

categories = "catering.restaurant,commercial.food_and_drink,catering.fast_food,catering.food_court,catering.bar"
radius = "50"
max_locations_returned = "5"

all_restaurants_df = pd.DataFrame()

for _ , row in df.iterrows():

    time.sleep(0.5)
    longitude = str(row['longitude_interdata'])
    latitude = str(row['latitude_interdata'])

    url = f"https://api.geoapify.com/v2/places?categories={categories}&filter=circle:{longitude},{latitude},{radius}&bias=proximity:{longitude},{latitude}&limit={max_locations_returned}&apiKey={geoapify_key}"

    payload = {}
    headers = {}

    response = requests.request("GET", url, headers=headers, data=payload)

    res = json.loads(response.text)
    # print(res)
    try:
        local_restaurant_df = pd.DataFrame([
            {
                **feature["properties"],
                "geo_lon": feature["geometry"]["coordinates"][0],
                "geo_lat": feature["geometry"]["coordinates"][1],
            }
            for feature in res["features"]
        ])
        
        # selected_columns = ['name', 'county', 'city', 'postcode', 'district', 'suburb', 'quarter', 'street', 'housenumber','lon','lat', 'formatted', 'catering']
        # local_restaurant_df = local_restaurant_df[selected_columns]

        local_restaurant_df['inspection_job_id'] = row['inspection_type_interdata']
        local_restaurant_df['job_id_interdata'] = row['job_id_interdata']
        local_restaurant_df['job_progress_interdata'] = row['job_progress_interdata']
        local_restaurant_df['house_number_interdata'] = row['house_number_interdata']
        local_restaurant_df['street_name_interdata'] = row['street_name_interdata']
        local_restaurant_df['address_interdata'] = row['address_interdata']
        local_restaurant_df['zip_code_interdata'] = row['zip_code_interdata']
        local_restaurant_df['latitude_interdata'] = row['latitude_interdata']
        local_restaurant_df['longitude_interdata'] = row['longitude_interdata']
        local_restaurant_df['result_interdata'] = row['result_interdata']
        local_restaurant_df['approved_date_interdata'] = row['approved_date_interdata']
        local_restaurant_df['nta_interdata'] = row['nta_interdata']
        
        all_restaurants_df = pd.concat([all_restaurants_df, local_restaurant_df])
    except Exception as e:
        # print(e)
        # Move on if no data is found
        continue
    

{'type': 'FeatureCollection', 'features': []}
{'type': 'FeatureCollection', 'features': []}
{'type': 'FeatureCollection', 'features': []}
{'type': 'FeatureCollection', 'features': []}
{'type': 'FeatureCollection', 'features': []}
{'type': 'FeatureCollection', 'features': []}
{'type': 'FeatureCollection', 'features': [{'type': 'Feature', 'properties': {'name': 'Hello, Yam!', 'country': 'United States', 'country_code': 'us', 'state': 'New York', 'county': 'New York County', 'city': 'New York', 'postcode': '10009', 'district': 'East Village', 'suburb': 'Manhattan', 'street': 'East 9th Street', 'housenumber': '4432', 'iso3166_2': 'US-NY', 'lon': -73.9829136, 'lat': 40.727499, 'state_code': 'NY', 'formatted': 'Hello, Yam!, 4432 East 9th Street, New York, NY 10009, United States of America', 'address_line1': 'Hello, Yam!', 'address_line2': '4432 East 9th Street, New York, NY 10009, United States of America', 'categories': ['catering', 'catering.fast_food', 'vegan'], 'details': ['details', 'd

In [ ]:
all_restaurants_df.to_csv('all_restaurants_df.csv', index = False)
# all_restaurants_df = pd.read_csv("all_restaurants_geoapify.csv")
# all_restaurants_df['approved_date_interdata'] = pd.Timestamp(all_restaurants_df['approved_date_interdata'])

selected_columns = [
    'name', 'county', 'city', 'postcode', 'district', 'suburb', 'housenumber', 'street', 'address_line2',
    'lon','lat', 'formatted', 'catering', 'commercial','house_number_interdata',
    'street_name_interdata', 'address_interdata', 'result_interdata', 'approved_date_interdata', 'nta_interdata'
    ]


geo_rest_df = all_restaurants_df.copy()
geo_rest_df['address_line2'] = geo_rest_df['address_line2'].apply(lambda x : x.split(',')[0].strip() if x else None)
geo_rest_df = all_restaurants_df[selected_columns]
geo_rest_df.head()

requested_places_saved = geo_rest_df.copy()

# Logic is fuzzy with addresses we try to pick ones that are close to matching then compare numerical addresses
requested_places_saved['ratio'] = requested_places_saved.apply(lambda x: fuzz.WRatio(x['address_line2'], x['address_interdata']), axis=1)
requested_places_saved = requested_places_saved[requested_places_saved['ratio'] >= 80]

requested_places_saved['split_check'] = requested_places_saved.apply(lambda x : x['housenumber'] == x['house_number_interdata'], axis = 1)
requested_places_saved = requested_places_saved[requested_places_saved['split_check'] == True]

# Column cleanup
requested_places_saved[['lon', 'lat']] = requested_places_saved[['lon', 'lat']].round(6)
selected_columns = [
    'name', 'county', 'city', 'postcode',
    'suburb', 'address_line2', 'address_interdata', 'lon', 'lat',
    'catering', 'commercial', 'approved_date_interdata', 'result_interdata', 'nta_interdata'
]
requested_places_saved = requested_places_saved[selected_columns]
requested_places_saved = requested_places_saved.rename(
    columns = {'address_line2' : 'address', 'approved_date_interdata' : 'approved_date', 'result_interdata' : "result", 'nta_interdata' : 'neighborhood'}
)

def category_handler(catering, commercial):
    if pd.notnull(catering): # If it is a restaurant it will show the type of catering
        try:
            # print(catering)
            return catering['cuisine'].title().strip()
        except Exception as e:
            # print("No correct cuisine found")
            return 'N/A'
    else: # If it is not a restaurant it will show what other type of place it is eg: bar/supermarket
        try:
            return commercial['type'].title().strip()
        except Exception as e:
            # print("No commercial type found")
            return 'N/A'
        
requested_places_saved['type'] = requested_places_saved.apply(lambda x : category_handler(x['catering'], x['commercial']), axis = 1)
requested_places_saved = requested_places_saved.drop(columns = ['catering', 'commercial'])

# Handle addresses
requested_places_saved['address'] = requested_places_saved['address'].apply(lambda x : x.split(',')[0])
requested_places_saved = requested_places_saved.drop(columns = ['address_interdata'])
        
requested_places_saved.to_csv("final_new_test_data.csv", index = False)

requested_places_saved


No commercial type found
No commercial type found
No commercial type found
No commercial type found
No commercial type found


,name,county,city,postcode,suburb,address,lon,lat,approved_date,result,neighborhood,type
0,Tipsy,Kings County,New York,11205,Brooklyn,584 Myrtle Avenue,-73.961238,40.693875,2026-03-02,Bait applied,Clinton Hill,Alcohol
1,CHF Restaurant,Kings County,New York,11220,Brooklyn,5411 5th Avenue,-74.012804,40.642499,2026-03-02,Bait applied,Sunset Park (Central),Chinese
0,Happy Garden,Kings County,New York,11207,Brooklyn,188 Wilson Avenue,-73.923511,40.699303,2026-03-04,Rat Activity,Bushwick (West),Chinese
0,Anand Indian Cuisine,New York County,New York,10075,Manhattan,304 East 78th Street,-73.955182,40.772368,2026-03-04,Rat Activity,Upper East Side-Lenox Hill-Roosevelt Island,Indian
0,Heidi’s House,New York County,New York,10075,Manhattan,308 East 78th Street,-73.955048,40.772330,2026-03-04,Rat Activity,Upper East Side-Lenox Hill-Roosevelt Island,American
2,Ed's Elbow Room,New York County,New York,10075,Manhattan,308 East 78th Street,-73.955017,40.772317,2026-03-04,Rat Activity,Upper East Side-Lenox Hill-Roosevelt Island,N/A
3,Orwashers,New York County,New York,10021,Manhattan,308 East 78th Street,-73.954944,40.772281,2026-03-04,Rat Activity,Upper East Side-Lenox Hill-Roosevelt Island,Bakery
1,New Kam Lai,New York County,New York,10024,Manhattan,514 Amsterdam Avenue,-73.975801,40.786924,2026-03-03,Bait applied,Upper West Side (Central),Chinese
0,The Heights Bar & Grill,New York County,New York,10025,Manhattan,2867 Broadway,-73.966402,40.805184,2026-03-03,Rat Activity,Morningside Heights,N/A
0,Native Noodles,New York County,New York,10032,Manhattan,2129 Amsterdam Avenue,-73.937872,40.838262,2026-03-03,Bait applied,Washington Heights (South),Asian;Singaporean


In [77]:
# requested_places_saved['catering'].iloc[0] is pd.notnull
# pd.notnull(None)
requested_places_saved.shape

(36, 13)

In [ ]:
# requested_places_saved = geo_rest_df.copy()

# requested_places_saved['ratio'] = requested_places_saved.apply(lambda x : fuzz.WRatio(x['formatted'], x['address_interdata']), axis = 1)
# requested_places_saved = requested_places_saved[requested_places_saved['ratio'] >= 80]

# # Check number in address for final comparison
# requested_places_saved['split_check'] = requested_places_saved.apply(lambda x : x['formatted'].split()[1] == x['address_interdata'].split()[0], axis = 1)
# # requested_places_saved = requested_places_saved[requested_places_saved['split_check'] == True]

# # # requested_places_saved = requested_places_saved.drop(columns = ["ratio", "split_check", "displayName", "formatted", "location"])

# # requested_places_saved[['lon', 'lat']] = requested_places_saved[['lon', 'lat']].round(6)

# requested_places_saved.to_csv("requested_places_saved.csv", index = False)
# requested_places_saved

import re
from rapidfuzz import fuzz

def normalise_address(address: str) -> str:
    if not isinstance(address, str):
        return ""
    address = address.lower().strip()
    address = re.sub(r'(\d+)(st|nd|rd|th)\b', r'\1', address)  # 5th → 5
    address = re.sub(r'\s+', ' ', address)                      # collapse whitespace
    return address

def addresses_match(formatted: str, interdata: str, threshold: int = 88) -> bool:
    parts = formatted.split(",")
    addr_segment = parts[1].strip() if len(parts) > 1 else formatted
    norm_formatted = normalise_address(addr_segment)
    norm_interdata = normalise_address(interdata)
    score = fuzz.token_sort_ratio(norm_formatted, norm_interdata)
    return score >= threshold

requested_places_saved = geo_rest_df.copy()

# Normalize addresses before comparing
# requested_places_saved['address_pull_normalized'] = requested_places_saved['formatted'].apply(lambda x : x.split(',')[1].strip() if x else None)
# requested_places_saved['address_interdata_normalized'] = requested_places_saved['address_interdata'].apply(lambda x : normalise_address(x))

requested_places_saved['ratio'] = requested_places_saved.apply(lambda x: fuzz.WRatio(x['address_line2'], x['address_interdata']), axis=1)
requested_places_saved = requested_places_saved[requested_places_saved['ratio'] >= 80]

requested_places_saved['split_check'] = requested_places_saved.apply(lambda x : x['housenumber'] == x['house_number_interdata'], axis = 1)
requested_places_saved = requested_places_saved[requested_places_saved['split_check'] == True]

# requested_places_saved['split_check'] = requested_places_saved.apply(lambda x: addresses_match(x['formatted'], x['address_interdata']), axis=1)

# requested_places_saved.to_csv("requested_places_saved.csv", index=False)
requested_places_saved

,name,county,city,postcode,district,suburb,housenumber,street,address_line2,lon,...,formatted,catering,commercial,house_number_interdata,street_name_interdata,address_interdata,result_interdata,approved_date_interdata,ratio,split_check
7,Tipsy,Kings County,New York,11205,NaN,Brooklyn,584,Myrtle Avenue,584 Myrtle Avenue,-73.961238,...,"Tipsy, 584 Myrtle Avenue, New York, NY 11205, ...",NaN,{'type': 'alcohol'},584,Myrtle Avenue,584 Myrtle Avenue,Bait applied,2026-03-02T13:36:26.000,100.000000,True
23,CHF Restaurant,Kings County,New York,11220,Sunset Park,Brooklyn,5411,5th Avenue,5411 5th Avenue,-74.012804,...,"CHF Restaurant, 5411 5th Avenue, New York, NY ...",{'cuisine': 'chinese'},NaN,5411,5 Avenue,5411 5 Avenue,Bait applied,2026-03-02T14:14:21.000,92.857143,True
35,Happy Garden,Kings County,New York,11207,NaN,Brooklyn,188,Wilson Avenue,188 Wilson Avenue,-73.923511,...,"Happy Garden, 188 Wilson Avenue, New York, NY ...",{'cuisine': 'chinese'},NaN,188,Wilson Avenue,188 Wilson Avenue,Rat Activity,2026-03-04T06:59:18.000,100.000000,True


In [43]:
# geo_rest_df

In [32]:
all_restaurants_df.to_csv("all_restaurants_geoapify.csv", index = False)